In [1]:
import pandas as pd
import numpy as np

In [2]:
events = pd.read_parquet(f"data/events.parquet")
races = pd.read_parquet(f"data/races.parquet")
runners = pd.read_parquet(f"data/runners.parquet")

In [3]:
fbase = "output/data"

# Pre-prep data

## Location Names

In [4]:
lmap = {
    "Aint": "Aintree",
    "Bang": "Bangor On Dee",
    "Bangor": "Bangor On Dee",
    "Bangor on Dee": "Bangor On Dee",
    "Bev": "Beverly",
    "Brig": "Brighton",
    "Carl": "Carlisle",
    "Cart": "Cartmel",
    "Catt": "Cattertick",
    "ChelmC": "Chelmsford City",
    "ChelmC PA": "Chelmsford City",
    "Chelt": "Cheltenham",
    "Chep": "Chepstow",
    "Chest": "Chester",
    "Donc": "Doncaster",
    "Epsm": "Epsom",
    "Extr": "Exeter",
    "Fake": "Fakenham",
    "FfosL": "Ffoss Las",
    "Font": "Fontwell",
    "GLghs": "Chelmsford City",
    "Good": "Goodwood",
    "Ham": "Hamilton",
    "Hayd": "Haydock",
    "Here": "Hereford",
    "Hex": "Hexham",
    "Hunt": "Huntingdon",
    "Kemp": "Kempton",
    "Leic": "Leicester",
    "Ling": "Lingfield",
    "Lud": "Ludlow",
    "MrktR": "Market Rasen",
    "Muss": "Musselburgh",
    "Newb": "Newbury",
    "Newc": "Newcastle",
    "Newm": "Newmarket",
    "Newt": "Newton Abbot",
    "Nott": "Nottingham",
    "Plump": "Plumpton",
    "Ponte": "Pontefract",
    "Redc": "Redcar",
    "Salis": "Salisbury",
    "Sand": "Sandown",
    "Sedge": "Sedgefield",
    "Sthl": "Southwell",
    "Strat": "Stratford",
    "Taun": "Taunton",
    "Uttox": "Uttoxeter",
    "Warw": "Warwick",
    "Weth": "Wetherby",
    "Winc": "Wincanton",
    "Wind": "Windsor",
    "Wolv": "Wolverhampton",
    "Worc": "Worcester",
    "Yarm": "Yarmouth"
}
events.location = events.location.replace(lmap)

## Normalised Probabilities

In [5]:
book_pct = runners.groupby("race_id").implied_odds.sum()
runners["normalised_odds"] = runners.implied_odds/runners.race_id.map(book_pct)

## Test Statistics

In [6]:
runners["returns"] = runners.bsp * runners.win_lose
runners["brier"]   = np.square(runners.normalised_odds - runners.win_lose)
runners["log_loss"] = -(
    np.log(runners.normalised_odds)*runners.win_lose +
    np.log(1-runners.normalised_odds)*(1-runners.win_lose)
)

/home/mabel/.pyenv/versions/3.14.2/lib/python3.14/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


## Bsp Binning Range

In [7]:
ticks = np.concat((
    np.arange(1, 2, 0.01),
    np.arange(2,3,0.02),
    np.arange(3,4,0.05),
    np.arange(4,6,0.1),
    np.arange(6,10,0.2),
    np.arange(10,20,0.5),
    np.arange(20,30,1),
    np.arange(30,50,2),
    np.arange(50,100,5),
    np.arange(100,1000,10),
    [1001]
))
tick_range=[0,2,3,4,6,10,20,30,50,100,1001]
tick_size=[0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1, 2, 5, 10]
#ticks=1000
t = {
    1: 0.01,
    2: 0.02,
    3: 0.05,
    4: 0.1,
    6: 0.2,
    10: 0.5,
    20: 1,
    30: 2,
    50: 5,
    100: 10
}

# Favourite-Longshot Bias

## General

In [8]:
g = runners.groupby(pd.cut(runners.bsp, bins=ticks))

In [9]:
all_df = pd.DataFrame({"mean": g.win_lose.mean(), "count": g.size(), "volume": g.pptradedvol.sum()})

In [10]:
all_df["tick_size"] = np.array(tick_size)[pd.cut(all_df.index.categories.right, bins=tick_range, labels=False)]
#df = df.rolling(5).mean()
all_df = all_df.rolling(5, min_periods=1).mean()
all_df["l"] = all_df.index.categories.left
all_df["r"] = all_df.index.categories.right
all_df.index=all_df.l

In [11]:
all_df.to_csv(f"{fbase}/all-fl.csv")

## Groups

per race stats

In [12]:
rg = runners.groupby("race_id")

nraces = races.assign(
    ppvol=rg.pptradedvol.sum(),
    nrunners=rg.size(),
    overround=rg.implied_odds.sum()
)

add groups that are needed

In [13]:
df = runners.join(nraces, on="race_id").join(events, on="event_id")

In [14]:
cats = {
    "race_no": ("race_no", [1,2,3,4,5,6,7,8]),
    "distance_f": ("distance_f", df.distance_f.value_counts().head(10).index),
    "hour": (df.race_time.dt.hour, np.arange(12,20,1)),
    "race_type": ("race_type", df.race_type.value_counts().head(10).index),
    "location": ("location", df.location.value_counts().head(10).index),
    "nrunners": ("nrunners", df.nrunners.value_counts().head(10).index),
    "ppvol": (df.race_id.map(pd.qcut(nraces.ppvol, q=10)), slice(None)),
    "overround": (df.race_id.map(pd.qcut(nraces.overround, q=10)), slice(None))
}

In [15]:
for c in cats:
    for a, v in [("mean", "win_lose"), ("count", "win_lose"), ("sum", "pptradedvol")]:
        pt = df.pivot_table(
            index=pd.cut(df.bsp, bins=ticks),
            columns=cats[c][0],
            values=v,
            aggfunc=a
        ).loc[:, cats[c][1]]
        pt = pt.reindex(sorted(pt.columns), axis=1)
        pt.columns = pt.columns.map(str).str.replace(",", ";")
        ts = np.array(tick_size)[pd.cut(pt.index.categories.right, bins=tick_range, labels=False)]
        pt.insert(0, 'tick_size', ts)
        #pt=pt.groupby("tick_size", as_index=False).rolling(5, min_periods=1).mean()
        pt = pt.rolling(5).mean()
        l = pt.index.categories.left
        r = pt.index.categories.right
        
        pt.index=l
        pt.to_csv(f"{fbase}/{c}-fl-{a}.csv")

# Test Statistics

## All

In [16]:
all_brier = runners.groupby("race_id").brier.mean().mean()
all_ll = runners.groupby("race_id").log_loss.mean().mean()

In [17]:
print(f"Brier: {all_brier}")
print(f"Log-Loss: {all_ll}")

Brier: 0.09511011721519749
Log-Loss: 0.3169538329161357


In [18]:
b = []
l = []
r = np.arange(0.02,1,0.02)
for x in r:
    b.append(np.square(x - runners.win_lose).groupby(runners.race_id).mean().mean())
    l.append(-(np.log(x)*runners.win_lose + np.log(1-x)*(1-runners.win_lose)).groupby(runners.race_id).mean().mean())
#d = pd.DataFrame(d)
ndf = pd.DataFrame({"brier": 1-b/all_brier, "log_loss": 1-l/all_ll}, index=r)
ndf.to_csv(f"{fbase}/all_ss_random.csv")

## Groups

In [19]:
for c in cats:
    b = df.groupby(["race_id", cats[c][0]]).brier.mean().groupby(level=1).mean()
    l = df.groupby(["race_id", cats[c][0]]).log_loss.mean().groupby(level=1).mean()

    f = pd.DataFrame({"brier": 1-b/all_brier,
                      "log_loss": 1-l/all_ll,
                      "count": df.groupby(["race_id", cats[c][0]]).size().groupby(level=1).size(),
                      "volume": df.groupby(cats[c][0]).pptradedvol.sum()})
    f = f.loc[cats[c][1], :]
    f = f.sort_index()
    f.to_csv(f"{fbase}/{c}-acc.csv")

In [20]:
df.groupby(["race_id", "race_no"]).size().groupby(level=1).sum()

race_no
1     227158
2     216066
3     216319
4     215345
5     220629
6     222976
7     190690
8      61719
9       6765
10       391
11        15
12        18
13        21
14        17
15         7
dtype: int64

In [21]:
pd.qcut(nraces.ppvol, q=10)

race_id
162764294    (325584.681, 386602.206]
162764320    (264075.638, 325584.681]
162764284    (511378.138, 583090.525]
162764331    (386602.206, 448046.625]
162764361      (583090.525, 676275.5]
                       ...           
186160695    (264075.638, 325584.681]
186161070    (264075.638, 325584.681]
186161164      (583090.525, 676275.5]
186160690    (448046.625, 511378.138]
186161060    (448046.625, 511378.138]
Name: ppvol, Length: 174779, dtype: category
Categories (10, interval[float64, right]): [(-0.001, 203392.788] < (203392.788, 264075.638] < (264075.638, 325584.681] < (325584.681, 386602.206] ... (511378.138, 583090.525] < (583090.525, 676275.5] < (676275.5, 825315.6] < (825315.6, 13317541.0]]

In [22]:
df.race_id.map(pd.qcut(nraces.ppvol, q=10))

selection_id
8709395     (325584.681, 386602.206]
20377712    (264075.638, 325584.681]
24209475    (511378.138, 583090.525]
24209478    (511378.138, 583090.525]
24148082    (386602.206, 448046.625]
                      ...           
40484789      (583090.525, 676275.5]
40262716      (583090.525, 676275.5]
35415057    (325584.681, 386602.206]
28625715    (325584.681, 386602.206]
11314105    (264075.638, 325584.681]
Name: race_id, Length: 1578136, dtype: category
Categories (10, interval[float64, right]): [(-0.001, 203392.788] < (203392.788, 264075.638] < (264075.638, 325584.681] < (325584.681, 386602.206] ... (511378.138, 583090.525] < (583090.525, 676275.5] < (676275.5, 825315.6] < (825315.6, 13317541.0]]

In [23]:
df.groupby(["race_id", df.race_id.map(pd.qcut(nraces.ppvol, q=10))]).size().groupby(level=1).size()

race_id
(-0.001, 203392.788]        17478
(203392.788, 264075.638]    17478
(264075.638, 325584.681]    17478
(325584.681, 386602.206]    17478
(386602.206, 448046.625]    17478
(448046.625, 511378.138]    17477
(511378.138, 583090.525]    17478
(583090.525, 676275.5]      17478
(676275.5, 825315.6]        17478
(825315.6, 13317541.0]      17478
dtype: int64

In [24]:
df.groupby(df.race_id.map(pd.qcut(nraces.ppvol, q=10))).size()

race_id
(-0.001, 203392.788]        154730
(203392.788, 264075.638]    154058
(264075.638, 325584.681]    159219
(325584.681, 386602.206]    163607
(386602.206, 448046.625]    163460
(448046.625, 511378.138]    160941
(511378.138, 583090.525]    156845
(583090.525, 676275.5]      152516
(676275.5, 825315.6]        149411
(825315.6, 13317541.0]      163349
dtype: int64

In [25]:
nraces.groupby(pd.qcut(nraces.ppvol, q=10)).size()

ppvol
(-0.001, 203392.788]        17478
(203392.788, 264075.638]    17478
(264075.638, 325584.681]    17478
(325584.681, 386602.206]    17478
(386602.206, 448046.625]    17478
(448046.625, 511378.138]    17477
(511378.138, 583090.525]    17478
(583090.525, 676275.5]      17478
(676275.5, 825315.6]        17478
(825315.6, 13317541.0]      17478
dtype: int64

In [26]:
df[

_IncompleteInputError: incomplete input (3910388838.py, line 1)